# Case Study 2: The G10 FX Carry Trade

**Name:**
**Data as-of date:** *(TODO: fill in after your final run; FRED/Yahoo update over time)*

Skeleton notebook following the assignment's Sections 0–6. Everything marked `TODO` is yours to fill in.
Cells marked **Checkpoint** are sanity checks: run them and make sure the output looks reasonable before moving on.

| Section | Points |
|---|---|
| 1. Data pipeline | 15 |
| 2. Modules 3–4, basket-wide | 15 |
| 3. Modules 5–6, basket-wide | 20 |
| 4. Basket construction | 25 |
| 5. Backtest and tearsheet | 25 |
| 6. Bonus: trend overlay | +5 |

## Rookie-Mistake Checklist

Tick these off as you go. Each one appears again in the section where it's most likely to bite.

- [ ] **Trade direction / sign.** One quoting convention for every spot series (Section 1.2).
- [ ] **Accrual convention.** One convention, documented, never switched (Section 1.4).
- [ ] **Risk-adjusted sizing**, not raw-yield ranking alone (Section 4.2).
- [ ] **No look-ahead in vol forecasts.** The weight held on day $t+1$ uses information through day $t$ (Sections 4.2, 5.1).
- [ ] **Funding spread** included (Section 4.3).
- [ ] **Correlation of legs** checked; ten currencies is not ten independent bets (Section 4.4).
- [ ] **Skewness, kurtosis, and drawdown** reported, not just Sharpe (Section 5.2).
- [ ] **Reproducible cache** with a documented as-of date (Section 1.3).

---
# 0. Before You Start: What's Already Built

Read the course's `fetch_data.py` first. Note down:
- which FRED series and Yahoo tickers it pulls,
- how it aligns monthly rates to daily FX dates,
- exactly how it computes `carry_pnl` (accrual convention, units).

**TODO (prose):** Summarize in 3–5 bullets what `fetch_data.py` does and what you'll change to extend it to G10.

In [ ]:
import os, sys, time, warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import yfinance as yf
import pandas_datareader.data as web          # or fredapi; match whatever fetch_data.py uses

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
from arch import arch_model

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

RNG = np.random.default_rng(5005)              # one seed for the whole notebook (bootstrap, etc.)
CACHE_DIR = Path("data_cache"); CACHE_DIR.mkdir(exist_ok=True)
START = "2000-01-01"                           # TODO: choose; make sure it covers 2008 if possible

CCYS     = ["EUR", "JPY", "GBP", "CHF", "CAD", "AUD", "NZD", "SEK", "NOK"]   # the nine non-USD currencies
ALL_CCYS = ["USD"] + CCYS

---
# 1. Data Pipeline: Building the G10 Universe (15 pts)

## 1.1 Short-Term Interest Rates

**Verify every series ID on fred.stlouisfed.org before using it.** For each one, check:
1. that it exists and is still updated (look at the last observation date),
2. its frequency (monthly vs. annual),
3. its units (percent per annum).

Fill in the dictionary below only after checking.

| Currency | FRED series | Frequency | Last obs | Notes / substitutions |
|---|---|---|---|---|
| USD | | | | |
| EUR | | | | Euro Area aggregate |
| JPY | `IRSTCI01JPM156N` | | | from lecture |
| GBP | | | | |
| CHF | | | | |
| CAD | | | | may need `IR3TIB01...` family |
| AUD | `IRSTCI01AUM156N` | | | from lecture |
| NZD | | | | |
| SEK | | | | |
| NOK | | | | |

**TODO (prose):** Justify each substitution. If you mix call-money and 3-month interbank rates, say why that's acceptable (or what bias it introduces).

**Think about this:** OECD monthly rates are monthly *averages*, published after the month ends. If you forward-fill March's value onto March's trading days, are you using information you wouldn't have had on March 3rd? Decide whether to lag the rates by one month and document the choice.

In [ ]:
FRED_IDS = {
    "USD": None,                 # TODO
    "EUR": None,                 # TODO
    "JPY": "IRSTCI01JPM156N",    # verify
    "GBP": None,                 # TODO
    "CHF": None,                 # TODO
    "CAD": None,                 # TODO
    "AUD": "IRSTCI01AUM156N",    # verify
    "NZD": None,                 # TODO
    "SEK": None,                 # TODO
    "NOK": None,                 # TODO
}

## 1.2 Spot FX Rates

**Pick one convention and use it everywhere.** A natural choice: $S_{i,t}$ = **USD per 1 unit of currency $i$**. Then:
- $\Delta s_{i,t} = \log S_{i,t} - \log S_{i,t-1} > 0$ means currency $i$ **appreciated** against USD,
- which is a **gain** for a position that is long currency $i$ and short USD.

Some Yahoo tickers are quoted the other way (USD/XXX, e.g. USDJPY), so those need to be **inverted**.

| Currency | Yahoo ticker | Quote direction | Invert? |
|---|---|---|---|
| EUR | `EURUSD=X` | USD per EUR | no |
| JPY | | | |
| GBP | | | |
| CHF | | | |
| CAD | | | |
| AUD | | | |
| NZD | | | |
| SEK | | | |
| NOK | | | |

*(Verify each ticker on Yahoo and confirm its direction by eye: e.g. USD per JPY should be about 0.007, not 150.)*

In [ ]:
YF_TICKERS = {
    # ccy: (ticker, invert)  -- invert=True if the ticker is quoted as USD/XXX
    "EUR": ("EURUSD=X", False),
    # TODO: the other eight
}

## 1.3 The Cache

Design goals:
- Read from disk if the cached file exists **and** covers the latest available date.
- Otherwise fetch (ideally only the missing tail), merge, and write back.
- Never silently serve stale data: print what was loaded and its last date.

**TODO (prose):** Describe your cache-invalidation rule. What counts as "the most recent available date" for a monthly FRED series vs. a daily Yahoo series? (They differ.) What happens on a weekend or holiday?

In [ ]:
def load_or_fetch(name, fetch_fn, max_staleness_days, force=False):
    '''
    name               : cache file stem, e.g. "rates" or "spot"
    fetch_fn           : function(start_date) -> DataFrame with a DatetimeIndex
    max_staleness_days : how old the last row may be before refetching (e.g. ~45 for monthly, ~3 for daily)
    '''
    path = CACHE_DIR / f"{name}.csv"
    # TODO:
    #   1. if the file exists and not force: read it
    #   2. decide whether it's stale by comparing its last index date with today
    #   3. if stale or missing: fetch, merge with cached rows (drop duplicate dates), save
    #   4. print a one-line summary: rows, first date, last date, fetched or cached
    ...

def fetch_rates(start):
    # TODO: pull every FRED_IDS series, return one DataFrame (columns = currencies, in % p.a.)
    ...

def fetch_spot(start):
    # TODO: pull every YF_TICKERS series (Close), invert where needed,
    #       return one DataFrame (columns = currencies, USD per unit of currency)
    ...

In [ ]:
# rates = load_or_fetch("rates", fetch_rates, max_staleness_days=...)
# spot  = load_or_fetch("spot",  fetch_spot,  max_staleness_days=...)

# Checkpoint: shapes, date ranges, missing values per column, and one sanity value per currency
# print(rates.tail()); print(spot.tail()); print(spot.isna().sum())

## 1.4 Constructing Per-Currency Carry P&L

For a position **long currency $i$, funded in USD**, from day $t-1$ to day $t$:

$$
z_{i,t} \approx \underbrace{\Delta s_{i,t}}_{\text{spot log return}} + \underbrace{(i_{i,t-1} - i_{\text{USD},t-1}) \cdot \frac{d_t}{365}}_{\text{accrued differential}}
$$

where $d_t$ is the number of **calendar days** between the two observations. Interest accrues over weekends, so Monday's row has $d_t = 3$.

Decisions to make and document:
1. **Simple vs. compounded accrual.** The simple version is above; the compounded version is $\log(1 + i\, d_t/365)$ per leg. Pick one.
2. **Which day's rate** applies to the accrual? It should be the rate known at $t-1$ (when you were holding the position).
3. **NaN handling.** Holidays differ across countries. Do you drop any day with a missing value, or forward-fill spot? Forward-filling spot creates a fake zero return followed by a two-day return. Is that acceptable?

**Units:** keep everything in **percent** (×100), matching the lectures.

In [ ]:
# TODO: align rates (monthly) onto the spot index (daily) -> rates_daily
#       remember the publication-lag decision from 1.1
rates_daily = ...

# TODO: spot log returns, in %
spot_ret = ...

# TODO: calendar-day gaps between consecutive rows
d = ...   # e.g. pd.Series(spot_ret.index, index=spot_ret.index).diff().dt.days

# TODO: accrued differential vs USD for each currency, using the PREVIOUS day's rates
carry_accrual = ...

# TODO: total carry P&L DataFrame, one column per non-USD currency
carry = ...

In [ ]:
# Checkpoint
# 1. carry.shape, carry.index.min(), carry.index.max(), carry.isna().sum()
# 2. Sign check: on a day when EUR clearly strengthened vs USD, carry["EUR"] should be positive
# 3. Accrual check: carry_accrual["AUD"].mean()*365 should be roughly the average AUD-USD rate gap
# 4. Continuity: compare your AUD and JPY legs against the lecture's carry_pnl.csv
#    (AUD/JPY carry is approximately carry["AUD"] - carry["JPY"]; why?)

---
# 2. Revisiting Modules 3–4: Is the Single-Pair Story Robust? (15 pts)

## 2.1 Random Walk / Stationarity Check, Basket-Wide

Two complementary pieces of evidence for each of the nine spot series (the USD column is the numeraire):
- **Visual:** SACF of log price vs. SACF of log returns (Module 4, Part 5).
- **Formal:** an ADF test (`adfuller`) on log price and on log returns. The null is a unit root (random walk), so a small p-value means stationary.

**Think about:** CHF had a floor against EUR (2011–2015) and DKK-style pegs exist in the G10 neighborhood. What would a managed or pegged currency look like in these diagnostics?

In [ ]:
# TODO: grid of SACF plots, e.g. 9 rows x 2 cols (log price | log return), or two 3x3 grids

In [ ]:
# TODO: summary table: currency | ADF stat (log price) | p-value | ADF stat (log return) | p-value
adf_table = ...

**Interpretation (TODO):** Do all log prices look like random walks? Any exceptions, and when?

## 2.2 Mean Carry Return: Basket-Wide Significance

Module 4 IID test for each currency: $\bar z$, $\text{SE} = s/\sqrt{n}$, $t = \bar z / \text{SE}$.

Also report the **average rate differential** per currency, so you can check the forward-premium-puzzle question: do the highest-differential currencies have the most robust positive carry?

In [ ]:
# TODO: table: ccy | mean daily carry (%) | SE | t-stat | annualized mean (%) | avg differential vs USD (% p.a.)
mean_table = ...

In [ ]:
# TODO (recommended): scatter of avg differential (x) vs annualized mean carry (y), labeled by currency

**Interpretation (TODO):** Which currencies are significant? Does carry P&L line up with the differential? Connect this to UIP and the forward premium puzzle.

## 2.3 Bootstrap Robustness

Pick 2–3 currencies, including AUD/JPY for continuity. Resample **days with replacement** 500–1000 times and record the mean each time.

**Think about:** the IID bootstrap assumes days are exchangeable. Given volatility clustering (Module 6), is that assumption fair? (A block bootstrap is a natural extension, worth a sentence even if you don't implement it.)

In [ ]:
def bootstrap_mean(x, B=1000, rng=RNG):
    # TODO: return an array of B bootstrap sample means
    ...

# TODO: AUD/JPY series for continuity (the lecture CSV, or carry["AUD"] - carry["JPY"])
# TODO: histograms of bootstrap means, with a vertical line at 0 and at the sample mean
# TODO: report the share of bootstrap means <= 0 and a 95% percentile interval

**Interpretation (TODO):** Is the AUD/JPY finding robust or fragile? How does it compare to the other currencies you picked?

---
# 3. Revisiting Modules 5–6: Mean Predictability and Volatility per Currency (20 pts)

## 3.1 ARMA Identification, Basket-Wide
For each currency: ACF/PACF, then an AIC/BIC grid over $p, q \in \{0,\dots,3\}$ (Module 5).

In [ ]:
def arma_grid(x, max_p=3, max_q=3):
    # TODO: return a DataFrame of (p, q, AIC, BIC); use try/except around each fit
    ...

# TODO: loop over currencies, keep the best order by BIC (and by AIC), and collect a summary table:
#       ccy | best (p,q) by BIC | best (p,q) by AIC | any AR/MA coefficient significant?
arma_summary = ...

In [ ]:
# TODO: compact ACF/PACF grid for all nine currencies

**Interpretation (TODO):** Does "no meaningful mean predictability" hold basket-wide? Any outliers? Is a significant coefficient also *economically* meaningful?

## 3.2 GARCH Fitting, Basket-Wide

For each currency, fit GARCH(1,1) with Normal and with Student-$t$ innovations.

**Reminders from Module 6:**
- `res.conditional_volatility` is already a standard deviation.
- Implied unconditional variance is $\hat\alpha_0 / (1 - \hat\alpha_1 - \hat\beta_1)$.
- These are **full-sample** fits: fine for description here, but they contain look-ahead bias if used to trade (that's what Section 5.1 fixes).

In [ ]:
garch_fits = {}   # ccy -> {"normal": res, "t": res}

# TODO: loop over currencies, fit both, store them

In [ ]:
# TODO: summary table:
#   ccy | alpha1 | beta1 | persistence | nu (t fit) | AIC normal | AIC t | BIC normal | BIC t | winner
#       | Ljung-Box p-value on std_resid**2 (e.g. lag 10)
garch_summary = ...

**Interpretation (TODO):** Do all nine show volatility clustering? Which are most persistent? Does the $t$ distribution win everywhere? Any fits with leftover dependence?

## 3.3 The Fama Regression, Basket-Wide

$$
\Delta s_{t+1} = \alpha + \beta\,(i_{\text{foreign},t} - i_{\text{USD},t}) + \varepsilon_{t+1}
$$

**Before coding, derive the UIP prediction for $\beta$ under *your* quoting convention.** UIP says the expected return on the carry position is zero. With $S$ = USD per unit of foreign currency, work out $E_t[\Delta s_{t+1}]$ in terms of $i_{\text{foreign}} - i_{\text{USD}}$ and check whether UIP implies $\beta = +1$ or $\beta = -1$ in this regression. If it doesn't match the handout, either flip the regressor or the spot convention, and say so explicitly. This is exactly the sign-error trap.

**Practical choices to document:**
- **Units must match.** Either use daily $\Delta s$ with the differential converted to a daily rate, or (often cleaner, since the rates are monthly) sample both at **monthly** frequency with the annual differential divided by 12.
- **Standard errors.** Residuals are heteroskedastic (GARCH!). Consider HAC (Newey–West) SEs: `.fit(cov_type="HAC", cov_kwds={"maxlags": ...})`.
- The interesting test is $H_0: \beta = 1$ (UIP), not just $\beta = 0$.

In [ ]:
# TODO: for each currency, build y (spot change) and x (lagged differential) at your chosen frequency,
#       run OLS with HAC SEs, and collect:
#       ccy | beta_hat | SE | t-stat for beta=0 | t-stat for beta=1 (UIP) | n
fama_table = ...

**Interpretation (TODO):** Does the forward premium puzzle show up basket-wide? Explain why this table justifies expecting positive P&L from a cross-sectional carry basket.

---
# 4. Building the G10 Carry Basket (25 pts)

**Shared machinery.** Every basket in Sections 4–6 can be written as a daily weight matrix $W$ (dates × 9 currencies), with daily P&L

$$
r^{\text{basket}}_{t} = \sum_i w_{i,t-1}\, z_{i,t}.
$$

A negative weight is a short position in currency $i$ vs. USD. It loses the differential and gains if $i$ depreciates, which the sign handles automatically.
Building $W$ once and reusing one P&L/metrics function keeps all strategies comparable.

**Simplification to note:** holding *weights* fixed for a month implicitly assumes daily rebalancing to those weights; holding *notionals* fixed lets weights drift. Either is acceptable if you document it.

In [ ]:
def month_start_rebalance_dates(index):
    # TODO: first trading day of each month in `index`
    ...

def basket_pnl(W, carry):
    '''W: weights decided at each date's close, applied to the NEXT day's return.'''
    # TODO: (W.shift(1) * carry).sum(axis=1), with NaN handling
    ...

def perf_stats(r, freq=252):
    # TODO: dict with ann. return, ann. vol, Sharpe, skew, excess kurtosis, max drawdown
    ...

## 4.1 Rank-and-Sort Basket
At each month start: rank the nine currencies by the differential vs USD **known at that date**. Long the top 3 (+1/6 each), short the bottom 3 (−1/6 each), so the long notional equals the short notional (+0.5 / −0.5 gross per side; choose and document your scaling).

In [ ]:
# TODO: build W_rank (dates x ccys): set weights on rebalance dates, then forward-fill until the next one
W_rank = ...
pnl_rank = ...

# Checkpoint: W_rank.sum(axis=1) should be ~0 (dollar-neutral); (W_rank != 0).sum(axis=1) should be 6

In [ ]:
# TODO: heatmap or table of which currencies were long/short over time (shows the basket's composition drift)

## 4.2 Volatility-Targeted (Risk-Parity) Basket

Within each leg, weight $\propto 1/\hat\sigma_{i}$, then rescale so each leg still sums to ±0.5 (or to a target volatility; document the choice).

**Timing:** the weight held over day $t+1$ must use $\hat\sigma_{i,t+1|t}$, which is built from returns **through day $t$** only.
- With `arch`, `res.conditional_volatility[t]` is $\sigma_{t|t-1}$: the volatility *for* day $t$, built from data through $t-1$.
- Write out which index of `conditional_volatility` (or `res.forecast(...)`) corresponds to "known at the close of day $t$, used for day $t+1$," then check that `basket_pnl`'s `shift(1)` isn't shifting a second time.

*(Using the full-sample fits from 3.2 here still leaks parameter information. Note it; Section 5.1 removes it.)*

In [ ]:
# TODO: sigma_fcst DataFrame (dates x ccys) of one-step-ahead vol forecasts, aligned so row t = known at close of t
sigma_fcst = ...

# TODO: build W_vol from W_rank's long/short membership, using inverse-vol weights within each leg
W_vol = ...
pnl_vol = ...

# Checkpoint: leg sums still +0.5 / -0.5; no NaNs in the first rows after the warm-up

## 4.3 Adding Realism: Costs and Funding Spread

- **Transaction cost:** at each rebalance, turnover $= \sum_i |w_{i,\text{new}} - w_{i,\text{old}}|$ and cost $= \text{bps}/10^4 \times \text{turnover}$, charged on the rebalance day.
- **Funding spread:** each long leg earns (differential − spread); each short leg pays (differential + spread). Either way you lose $\text{spread} \times d_t/365$ per unit of **gross** exposure per day.

Run a small grid, e.g. costs ∈ {0, 5, 10} bps and spread ∈ {0, 10, 25} bps p.a.

In [ ]:
def apply_frictions(W, carry, d, cost_bps, spread_bps_pa):
    # TODO: return net daily P&L = gross P&L - transaction costs - funding drag
    ...

# TODO: grid table of net Sharpe / ann. return for both baskets across (cost_bps, spread_bps_pa)
friction_grid = ...

**Interpretation (TODO):** How much gross carry survives realistic frictions? Which basket is more sensitive, and why (turnover)?

## 4.4 Basket vs. Single Pair
Same sample period for all three: (a) AUD/JPY single pair, (b) equal-weight rank-and-sort, (c) vol-targeted.

In [ ]:
# TODO: perf_stats table for the three strategies (common date range!)
# TODO: cumulative P&L plot of the three

In [ ]:
# TODO: correlation matrix (heatmap) of the nine carry legs, and optionally a rolling average pairwise correlation
#       (addresses the "ten currencies != diversified" rookie mistake)

**Interpretation (TODO):** Does the basket improve risk-adjusted returns vs. the single pair, and by how much? What does the correlation structure say about real diversification?

---
# 5. Full Backtest and Risk Tearsheet (25 pts)

## 5.1 Walk-Forward Loop

At each monthly rebalance date $t$, using **only data through $t$**:
1. rank currencies by the differential known at $t$,
2. refit GARCH on an expanding window **if** a refit is due (choose a frequency: monthly, quarterly, or annually, and justify it),
3. between refits, update $\hat\sigma$ daily with the variance recursion and the *fixed* most recent parameters,
4. compute the vol-targeted weights, trade into them, and charge costs and funding spread.

**Choices to document:** initial training window (e.g. 2–3 years), refit frequency, and the `arch` approach. Useful tools: `arch_model(...).fit(last_obs=date)` fits only on data before `last_obs`; `res.forecast(start=...)` produces forecasts beyond it with those parameters.

**Checkpoint idea:** add an `assert` that every fit's training data ends on or before its rebalance date.

In [ ]:
REFIT_EVERY = ...      # TODO, e.g. "QS" (quarterly) and justify
WARMUP_YEARS = ...     # TODO
COST_BPS, SPREAD_BPS_PA = ..., ...   # TODO: your base-case frictions

def walk_forward(carry, rates_daily, d):
    '''
    Returns: W_wf (dates x ccys), net daily P&L, turnover series, costs series, sigma forecasts used
    '''
    # TODO: loop over rebalance dates
    #   - skip until warm-up is complete
    #   - if a refit is due: refit GARCH-t per currency on data[:t] and store params
    #   - get sigma_{t+1|t} for each currency using params fit on data through t
    #   - rank, build inverse-vol leg weights
    #   - record weights; costs on turnover
    ...

# W_wf, pnl_wf, turnover_wf, costs_wf, sigma_wf = walk_forward(carry, rates_daily, d)

In [ ]:
# Checkpoint: compare pnl_wf to the in-sample vol-targeted pnl_vol (Section 4.2) over the same dates.
# A large gap in favor of the in-sample version is evidence of how much look-ahead was flattering it.

## 5.2 Performance Report: The Tearsheet

**VaR note:** to get a *basket-level* 95% conditional VaR, the simplest route is a GARCH-$t$ on the basket's own P&L, walk-forward like 5.1. `arch`'s Student-$t$ is standardized to unit variance, so rescale the `scipy` quantile by $\sqrt{(\nu-2)/\nu}$.

In [ ]:
# TODO: walk-forward one-step-ahead 95% VaR series for the basket
var95 = ...

# TODO: tearsheet dict / DataFrame with every metric in the table below
#   turnover annualized = total turnover / years;  costs as % of initial capital
tearsheet = ...

| Metric | Value |
|---|---|
| Annualized return | |
| Annualized volatility | |
| Sharpe ratio | |
| **Skewness** | |
| **Excess kurtosis** | |
| Maximum drawdown | |
| 95% conditional VaR (GARCH-$t$) | |
| Total turnover (annualized) | |
| Total costs + funding spread paid (% of initial capital) | |

*(TODO: fill in from the cell above, or render the DataFrame as markdown programmatically.)*

In [ ]:
# TODO: naive benchmark: long every currency equally vs USD, no ranking (W = 1/9 each)
# TODO: cumulative P&L plot: walk-forward basket vs AUD/JPY vs naive benchmark (labels, legend, title)
# TODO: drawdown plot underneath

In [ ]:
# TODO: VaR backtest: share of days where the loss exceeded the VaR forecast (target ~5%); plot P&L with -VaR band

### Crash-episode zoom-in
Pick an episode inside your sample (2008 Q3–Q4; Aug 2024 JPY unwind; Jan 2015 SNB floor removal for CHF).
Plot over that window: basket P&L / drawdown and the GARCH VaR forecast. Did the risk estimate rise *before* the losses or only after?

In [ ]:
CRASH_START, CRASH_END = ..., ...   # TODO
# TODO: 2-panel zoom: cumulative P&L (or drawdown) | daily P&L with -VaR line

### PM Recommendation (TODO, 3–5 sentences)

## 5.3 Interpretation (TODO)

- **Vol targeting:** real reduction in drawdown / tail risk vs. equal-weight, or cosmetic? *(cite numbers)*
- **Frictions:** how much did costs and funding spread change the conclusion vs. Sections 4.1/4.2?
- **"Steady edge" or "nickels in front of a steamroller"?** Defend it with skewness, kurtosis, and the crash zoom.
- **First change you'd make running this book**, and why.

---
# 6. Bonus: Trend Overlay (up to +5)

Take the carry-implied position in currency $i$ only if trend agrees:
- long only if spot is above its trailing moving average (3m or 12m; pick and justify),
- short only if spot is below it,
- otherwise flat (or reduced size; justify).

The trend signal must also be computed with data through the rebalance date only.

In [ ]:
# TODO: trend signal at rebalance dates (dates x ccys), +1 / -1
# TODO: W_trend = W_wf masked where sign(W_wf) != trend signal; re-normalize legs (or not, and say why)
# TODO: tearsheet comparison with vs without overlay, especially skew, max DD, and the crash window

**Interpretation (TODO):** Did trend help, hurt, or mostly trade average return for a smoother path?

---
# Appendix: Reproducibility

- Data as-of date and package versions (cell below).
- Cache files are not committed; delete `data_cache/` to force a fresh pull.

In [ ]:
import importlib
for pkg in ["numpy", "pandas", "statsmodels", "arch", "yfinance", "scipy"]:
    try:
        print(pkg, importlib.import_module(pkg).__version__)
    except Exception as e:
        print(pkg, "not found")
print("Run date:", date.today())